# Lumexa Number Prediction Model — End-to-End Walkthrough

This notebook walks through the full workflow of the Number Prediction Model project:
generating data, preparing it, training a linear regression model, evaluating it honestly
with a train/test split, visualizing the results, and using the model for inference.

Run the cells in order from top to bottom.

In [ ]:
# 1. Imports
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline

## Step 1: Generate a synthetic dataset

We simulate a dataset where `test_score` follows a roughly linear relationship with
`hours_studied`, plus realistic random noise, clipped to a valid 0-100 score range.

In [ ]:
NUM_SAMPLES = 200
NOISE_LEVEL = 6.0
RANDOM_SEED = 42
BASE_SCORE = 42.0
HOURS_COEFFICIENT = 5.1

rng = np.random.default_rng(RANDOM_SEED)
hours_studied = np.round(rng.uniform(0.0, 12.0, size=NUM_SAMPLES), 2)
noise = rng.normal(0.0, NOISE_LEVEL, size=NUM_SAMPLES)
test_scores = np.clip(BASE_SCORE + HOURS_COEFFICIENT * hours_studied + noise, 0, 100)
test_scores = np.round(test_scores, 1)

print(f'Generated {NUM_SAMPLES} samples.')
print('First 5 rows:')
for i in range(5):
    print(f'  hours_studied={hours_studied[i]:.2f}, test_score={test_scores[i]:.1f}')

## Step 2: Prepare features (X) and labels (y)

scikit-learn expects features `X` as a 2D array, so we reshape `hours_studied`.

In [ ]:
X = hours_studied.reshape(-1, 1)
y = test_scores

print('X shape:', X.shape)
print('y shape:', y.shape)

## Step 3: Split into training and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')

## Step 4: Train the linear regression model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f'Learned slope: {model.coef_[0]:.4f}')
print(f'Learned intercept: {model.intercept_:.4f}')
print(f'Formula: test_score = {model.intercept_:.2f} + {model.coef_[0]:.2f} * hours_studied')

## Step 5: Evaluate the model honestly (train vs. test metrics)

In [ ]:
train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)

train_mse = mean_squared_error(y_train, train_predictions)
test_mse = mean_squared_error(y_test, test_predictions)
train_r2 = r2_score(y_train, train_predictions)
test_r2 = r2_score(y_test, test_predictions)

print(f'Training MSE: {train_mse:.2f}  |  Training R^2: {train_r2:.3f}')
print(f'Test MSE:     {test_mse:.2f}  |  Test R^2:     {test_r2:.3f}')

gap = train_r2 - test_r2
print(f'\nR^2 gap (train - test): {gap:.3f}')
if test_r2 > 0.6 and gap < 0.15:
    print('VERDICT: Good fit. The model generalizes well to unseen data.')
elif gap >= 0.15:
    print('VERDICT: Possible overfitting.')
else:
    print('VERDICT: Possible underfitting.')

## Step 6: Visualize the results

In [ ]:
sort_order = np.argsort(X_test.flatten())
X_test_sorted = X_test[sort_order]
predictions_sorted = test_predictions[sort_order]

plt.figure(figsize=(9, 6))
plt.scatter(X_train, y_train, color='lightgray', label='Training data', alpha=0.7)
plt.scatter(X_test, y_test, color='steelblue', label='Test data (actual)', s=55)
plt.plot(X_test_sorted, predictions_sorted, color='darkorange', linewidth=2.5, label='Model prediction')
plt.xlabel('Hours Studied')
plt.ylabel('Test Score')
plt.title('Lumexa Number Prediction Model: Study Hours vs. Test Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 105)
plt.show()

## Step 7: Use the model for inference on new data

In [ ]:
new_hours = np.array([1.5, 4.0, 7.5, 10.0]).reshape(-1, 1)
new_predictions = np.clip(model.predict(new_hours), 0, 100)

for hours, prediction in zip(new_hours.flatten(), new_predictions):
    print(f'Hours studied: {hours:>5.1f}  ->  Predicted score: {prediction:6.1f}')

## Summary

This notebook reproduced the entire `src/` pipeline (`data_prep.py`, `train.py`,
`evaluate.py`, `visualize.py`, `predict.py`) interactively in one place. Try changing
`NOISE_LEVEL` above and re-running from Step 1 to see how noisier data affects the
training/test R^2 gap discussed in Lesson 7 of the Python & AI Foundations course.